# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [5]:
df = pd.read_csv("AviationData.csv", encoding='latin1')
# Quick overview (optional, but good to keep)
print("Shape:", df.shape)
df.head()

Shape: (88889, 31)


C:\Users\OMokera\AppData\Local\Temp\ipykernel_8396\3791132160.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("AviationData.csv", encoding='latin1')


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [6]:
# Convert Event.Date to datetime, coercing errors to NaT
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# Filter from 1983-01-01
df = df[df['Event.Date'] >= '1983-01-01'].copy()

print(f"Remaining rows after filtering to 1983+: {len(df)}")

Remaining rows after filtering to 1983+: 85289


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [7]:
# Columns for injury counts
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 
               'Total.Minor.Injuries', 'Total.Uninjured']

# Fill NaNs with 0
for col in injury_cols:
    df[col] = df[col].fillna(0)

# Total passengers on board
df['Total_On_Board'] = df[injury_cols].sum(axis=1)

# Total fatal or serious injuries
df['Total_Fatal_Serious'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']

# Fraction of passengers injured (fatal/serious)
df['Injury_Fraction'] = df['Total_Fatal_Serious'] / df['Total_On_Board']

# For accidents with no passengers (e.g., cargo, zero on board), set fraction to NaN (or 0)
df.loc[df['Total_On_Board'] == 0, 'Injury_Fraction'] = np.nan

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [8]:
df['Aircraft.damage'].value_counts(dropna=False)

Aircraft.damage
Substantial    61775
Destroyed      17575
NaN             3138
Minor           2682
Unknown          119
Name: count, dtype: int64

In [9]:
# Example: adjust if other values like 'WRITTEN OFF' appear
df['Destroyed'] = df['Aircraft.damage'].str.strip().str.lower() == 'destroyed'
# Optionally treat NaN as False
df['Destroyed'] = df['Destroyed'].fillna(False)

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [10]:
# See current unique makes and their frequencies
make_counts = df['Make'].value_counts()
print(make_counts.head(20))

# Cleaning steps (example – you may need more based on inspection)
# 1. Strip whitespace
df['Make'] = df['Make'].str.strip()

# 2. Replace common variations (e.g., 'BOEING' vs 'THE BOEING COMPANY')
# Here is a dictionary – expand as you see fit
make_mapping = {
    'BOEING': 'Boeing',
    'THE BOEING COMPANY': 'Boeing',
    'AIRBUS INDUSTRIE': 'Airbus',
    'AIRBUS': 'Airbus',
    'CESSNA': 'Cessna',
    'PIPER': 'Piper',
    'BEECH': 'Beechcraft',
    'BEECHCRAFT': 'Beechcraft',
    'MCDONNELL DOUGLAS': 'McDonnell Douglas',
    'DOUGLAS': 'McDonnell Douglas',
    'LOCKHEED': 'Lockheed',
    'LOCKHEED MARTIN': 'Lockheed Martin',
}
df['Make'] = df['Make'].replace(make_mapping)

# 3. Drop rows with missing Make
df = df.dropna(subset=['Make'])

# 4. Keep makes with at least 50 accidents after filtering
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

print(f"Remaining rows after cleaning Make: {len(df)}")

Make
Cessna               20892
Piper                11301
CESSNA                4922
Beech                 4067
PIPER                 2841
Bell                  2022
Boeing                1545
BOEING                1151
BEECH                 1042
Mooney                1036
Grumman                990
Robinson               923
Bellanca               824
Hughes                 742
Schweizer              612
BELL                   588
Air Tractor            577
Mcdonnell Douglas      521
Aeronca                458
Maule                  428
Name: count, dtype: int64
Remaining rows after cleaning Make: 69919


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [12]:
# Drop rows where Model is missing
df = df.dropna(subset=['Model'])

# Strip whitespace
df['Model'] = df['Model'].str.strip()

# Create unique identifier
df['make_model'] = df['Make'] + ' ' + df['Model']

# Optional: keep only make_model with enough occurrences (e.g., ≥5 for reliable stats)
model_counts = df['make_model'].value_counts()
valid_models = model_counts[model_counts >= 5].index
df = df[df['make_model'].isin(valid_models)]

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [14]:
# Engine type
df['Engine.Type'].value_counts(dropna=False)
# Standardise categories: 'Reciprocating' -> 'Piston', 'Turbojet' -> 'Jet', etc.
engine_map = {
    'Reciprocating': 'Piston',
    'Turbo Prop': 'Turboprop',
    'Turbojet': 'Jet',
    'Turbofan': 'Jet',
    'Electric': 'Electric',
    'None': 'None',
}
df['Engine.Type'] = df['Engine.Type'].replace(engine_map)

In [15]:
# weather condition
df['Weather.Condition'].value_counts()
# Already simple: VMC, IMC, UNK. Keep as is, maybe map UNK to 'Unknown'

Weather.Condition
VMC    53448
IMC     4599
UNK      618
Unk      168
Name: count, dtype: int64

In [16]:
# Number of engines
# Convert to integer, fill NaNs with median or 0
df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')
median_engines = df['Number.of.Engines'].median()
df['Number.of.Engines'].fillna(median_engines, inplace=True)
df['Number.of.Engines'] = df['Number.of.Engines'].astype(int)

In [17]:
# purpose of flight
df['Purpose.of.flight'].value_counts()
# Group rare categories into 'Other'
purpose_counts = df['Purpose.of.flight'].value_counts()
rare_purposes = purpose_counts[purpose_counts < 100].index
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace(rare_purposes, 'Other')

In [18]:
# broad phase of light
df['Broad.phase.of.flight'].value_counts()
# Standardise (e.g., 'TAKEOFF' -> 'Takeoff', 'LANDING' -> 'Landing')
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.capitalize()

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [19]:
missing_frac = df.isnull().mean()
cols_to_drop = missing_frac[missing_frac > 0.7].index
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns: {list(cols_to_drop)}")

Dropped 4 columns: ['Aircraft.Category', 'FAR.Description', 'Schedule', 'Air.carrier']


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [20]:
df.to_csv('aviation_accidents_cleaned.csv', index=False)
print("Cleaned data saved as 'aviation_accidents_cleaned.csv'")

Cleaned data saved as 'aviation_accidents_cleaned.csv'
